# Phase 3 — Controlled Missingness Injection and Failure Detection

## Experiment Objective

The objective of this experiment is to evaluate whether the SHML system can detect controlled missing-value failures in incoming data batches.

The selected monitoring feature is `BILL_AMT1`, representing the most recent bill statement amount.

Unlike the preliminary Phase 1 experiments, which introduced missingness across all predictor features, this experiment focuses exclusively on `BILL_AMT1`.

## Experimental Design

- **Dataset:** UCI Default of Credit Card Clients
- **Model:** Logistic Regression
- **Monitoring feature:** `BILL_AMT1`
- **Healthy training missingness:** 0%
- **Detection threshold:** More than 5 percentage points above the healthy reference
- **Missingness scenarios:** 0%, 5%, 15%, and 30%
- **Experimental data:** Held-out validation features
- **Random seed:** 42
- **Test set:** Reserved for final evaluation

## Hypothesis

The feature-specific missingness detector will flag batches containing more than 5% missing values in `BILL_AMT1`, while leaving healthy batches and batches at or below the threshold unflagged.

## Scope

This experiment evaluates data-quality failure detection. Model correction, recovery validation, and acceptance decisions will be addressed in Phase 4 and Phase 5.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
from pathlib import Path

data_dir = Path("data/working")
candidates = sorted(data_dir.glob("*credit*card*clients*.xls*"))
if not candidates:
    raise FileNotFoundError(f"No dataset file found under {data_dir}")

DATA_PATH = str(candidates[0])
df = pd.read_excel(DATA_PATH, header=1)
df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [3]:
target_column = "default payment next month"

X = df.drop(columns=["ID", target_column])
y = df[target_column]

# Reproduce the original l70/15/15 split
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

print(
    "Healthy BILL_AMT1 missingness:",
    X_train["BILL_AMT1"].isna().mean()
)


Training shape: (21000, 23)
Validation shape: (4500, 23)
Test shape: (4500, 23)
Healthy BILL_AMT1 missingness: 0.0


In [4]:
#Implementing Failure Injection 
def inject_feature_missingness(
    X_batch,
    feature,
    missing_rate,
    random_state = 42
):
    if feature not in X_batch.columns:
        raise ValueError(f"Unknown feature: {feature}")
    
    if not 0 <= missing_rate <=1:
        raise ValueError("missing_rate must be between 0 and 1")
    
    rng = np.random.default_rng(random_state)
    
    X_corrupted = X_batch.copy(deep = True)
    
    n_missing = int(round(len(X_corrupted) * missing_rate))
    
    selected_positions = rng.choice(
        len(X_corrupted),
        size = n_missing,
        replace=False
    )
    
    feature_col = X_corrupted.columns.get_loc(feature)
    X_corrupted.iloc[selected_positions, feature_col] = np.nan
    return X_corrupted

In [5]:

missingness_levels = [0.00, 0.05, 0.15, 0.30]

corrupted_batches = {}

for rate in missingness_levels:
    X_corrupted = inject_feature_missingness(
        X_batch=X_val,
        feature="BILL_AMT1",
        missing_rate=rate,
        random_state=42
    )

    corrupted_batches[rate] = X_corrupted

    actual_rate = X_corrupted["BILL_AMT1"].isna().mean()

    print(
        f"Injected: {rate:.0%} | "
        f"Observed BILL_AMT1 missingness: {actual_rate:.2%}"
    )


Injected: 0% | Observed BILL_AMT1 missingness: 0.00%
Injected: 5% | Observed BILL_AMT1 missingness: 5.00%
Injected: 15% | Observed BILL_AMT1 missingness: 15.00%
Injected: 30% | Observed BILL_AMT1 missingness: 30.00%


## Step 5.5 — Feature-Specific Missingness Detection

### Objective

The objective is to determine whether incoming validation batches exhibit abnormal missingness in `BILL_AMT1` compared with the healthy training reference.

### Detection Rule

The healthy training reference has 0% missingness in `BILL_AMT1`.

A failure is detected when:

**Current batch missingness − Healthy reference missingness > 5 percentage points**

The threshold is a preliminary, predefined monitoring rule selected during Phase 2.

### Expected Behavior

- 0% missingness: No failure detected.
- 5% missingness: No failure detected.
- 15% missingness: Failure detected.
- 30% missingness: Failure detected.

The detector identifies a data-quality problem. It does not independently establish that predictive performance has degraded.

Detection performance and the appropriateness of the threshold will be evaluated in subsequent experiments.

In [ ]:
# Detect
def detect_feature_missingness(
    X_batch,
    feature,
    reference_missingness = 0.0,
    threshold = 0.05
):
    if feature not in X_batch.columns:
        raise ValueError(f"feature '{feature}' is missing")
    
    current_missingness = X_batch[feature].isna().mean()
    
    missingness_increase = (
        current_missingness - reference_missingness
        
    )
    failure_detected = missingness_increase > threshold
    return {
        "feature": feature,
        "current_missingness": current_missingness,
        "reference_missingness": reference_missingness,
        "missingness_increase": missingness_increase,
        "threshold": threshold,
        "failure_detected": bool(failure_detected)
    }


In [7]:

detection_results = []

for rate, X_batch in corrupted_batches.items():

    result = detect_feature_missingness(
        X_batch=X_batch,
        feature="BILL_AMT1",
        reference_missingness=0.0,
        threshold=0.05
    )

    detection_results.append({
        "Injected Missingness": rate,
        "Observed Missingness": result["current_missingness"],
        "Missingness Increase": result["missingness_increase"],
        "Failure Detected": result["failure_detected"]
    })

detection_df = pd.DataFrame(detection_results)

display(detection_df)


,Injected Missingness,Observed Missingness,Missingness Increase,Failure Detected
0,0.00,0.00,0.00,False
1,0.05,0.05,0.05,False
2,0.15,0.15,0.15,True
3,0.30,0.30,0.30,True


## Phase 3 — Experimental Findings and Conclusion

### Experiment Summary

The Phase 3 experiment evaluated a feature-specific missingness detector using the UCI Default of Credit Card Clients dataset.

Four validation batches were generated by injecting missing values exclusively into `BILL_AMT1` at rates of 0%, 5%, 15%, and 30%.

The healthy training reference contained 0% missingness in `BILL_AMT1`. A preliminary alert threshold of more than 5 percentage points above the reference was used.

### Detection Results

| Missingness Rate | Failure Detected |
|---|---|
| 0% | No |
| 5% | No |
| 15% | Yes |
| 30% | Yes |

All four experimental scenarios produced the expected detection decisions.

### Key Findings

1. The detector correctly calculated feature-specific missingness in all four experimental batches.
2. The 15% and 30% missingness scenarios triggered alerts.
3. The 0% and 5% scenarios did not trigger alerts.
4. The detector's behavior was consistent with the predefined threshold.
5. The experiment demonstrated deterministic detection of a specific data-quality failure.

### Limitations

The detection threshold was selected as an initial experimental rule rather than statistically calibrated.

The experiment used one feature, one validation dataset, and one random seed.

Successful detection of missingness does not necessarily imply predictive-performance degradation.

The detector has not yet been evaluated against a broader range of healthy and corrupted batches, and its generalization to other features and datasets remains untested.

### Conclusion

Phase 3 successfully established a reproducible logistic regression baseline and demonstrated feature-specific missingness detection.

The healthy baseline provides the reference for measuring predictive-performance changes, while the missingness detector identifies when the selected feature exceeds the predefined monitoring threshold.

The next phase will connect detected failures to automated diagnosis and recovery actions.

### Next Phase

**Phase 4 — Automated Failure Diagnosis and Recovery**

The system will identify the detected failure, select an appropriate correction, and apply training-fitted median imputation to affected data batches. Recovery effectiveness and safety decisions will be evaluated in subsequent experiments.